In [14]:
import PyPDF2
import pandas as pd
import re

def extract_pdf_data(file_path):
    from PyPDF2 import PdfReader
    
    reader = PdfReader(file_path)
    text = ''
    for page in reader.pages:
        text += page.extract_text()
    
    # Extraindo Número do Pedido
    numero_pedido = re.search(r'Numero Pedido: (\d+)', text).group(1)
    
    # Extraindo CNPJ da Filial
    cnpj_filial = re.search(r'(\d{8}/\d{4}-\d{2}) FARMACIA VALE VERDE LTDA', text).group(1)
    
    # Extraindo Data de Emissão do Pedido
    data_emissao = re.search(r'Dt. Emissão Pedido: (\d{2}/\d{2}/\d{4})', text).group(1)
    
    # Extraindo Tabela de Itens
    table_start = text.find('GENESIO A MENDES & CIA LTDA')
    table_end = text.find('Tot. Produtos:')
    table_text = text[table_start:table_end]
    
    # Processar a tabela de itens
    items = []
    rows = table_text.split('\n')
    for row in rows:
        # Quebrar a linha por espaços
        parts = row.split()
        
        if len(parts) > 10:  # Linhas que parecem ser válidas
            vlr_total = parts[0]
            codigo = parts[1]
            qtd = parts[2]
            
            if parts[3] == 'x' and parts[4].isdigit():
                qtd = f"{parts[2]} {parts[3]} {parts[4]}"
                parts.pop(3)
                parts.pop(3)
            
            vlr_unit = parts[3]
            icms = parts[4]
            desc = parts[5]
            fabricante = parts[-2]
            ipi = parts[-1]
            und = parts[12]
            
            descricao = ' '.join(parts[6:-4])
            
            items.append([vlr_total, codigo, qtd, vlr_unit, icms, desc, fabricante, descricao, und, ipi])
    
    # Criar um DataFrame com os itens
    columns = ['Vlr Total', 'Código', 'Qtd.', 'Vlr Unit', 'Icms', 'Desc.%', 'Fabricante', 'Descrição', 'Und', 'IPI']
    df_items = pd.DataFrame(items, columns=columns)
    
    return numero_pedido, cnpj_filial, data_emissao, df_items

file_path = r"C:\Users\Nicolas\Downloads\123964 - PEDIDO GAM - KOLESTON - 28-03-2024.PDF"
numero_pedido, cnpj_filial, data_emissao, df_items = extract_pdf_data(file_path)

print(f'Número do Pedido: {numero_pedido}')
print(f'CNPJ da Filial: {cnpj_filial.replace("/","").replace(".","").replace("-","")}')
print(f'Data de Emissão do Pedido: {data_emissao}')
print('Tabela de Itens:')
print(df_items)

KeyboardInterrupt: 